# AI 视频生产平台 · Colab ComfyUI 省算力一键启动器

专为 **Google AI Pro（200 Compute Units / 月）** 打造的按需启动工作流：
1. **省钱选卡**：顶部菜单 `代码执行程序 → 更改运行时类型` 选 **L4 GPU**（约 1.8 单元/小时，够跑 100+ 小时）；重型视频再临时切 A100/G4。
2. **模型存 Google Drive**：模型权重统一存放在 `MyDrive/AI_Video_Platform_ComfyUI/models`，每次开机秒挂载、不重复下载。
3. **闲置自动停表**：默认连续 **10 分钟**无新队列任务自动释放运行时（`runtime.unassign()`），防止忘关机空烧计算单元。
4. **一键对接平台**：第 3 步会输出 `https://xxxx.trycloudflare.com`，可直接浏览器打开 ComfyUI，也可填进平台「模型中心 → 引擎管理 → ComfyUI」测试连通。

In [ ]:
#@title 1. 挂载 Google Drive 并初始化 ComfyUI（首次约 1–2 分钟，后续秒启）
import os
import shutil
import subprocess
from pathlib import Path
from google.colab import drive

DRIVE_ROOT = Path('/content/drive/MyDrive/AI_Video_Platform_ComfyUI')
COMFY_DIR = Path('/content/ComfyUI')

drive.mount('/content/drive')

model_subdirs = [
    'checkpoints', 'loras', 'upscale_models', 'vae',
    'controlnet', 'clip', 'clip_vision', 'unet', 'diffusion_models', 'text_encoders'
]
for sub in model_subdirs:
    (DRIVE_ROOT / 'models' / sub).mkdir(parents=True, exist_ok=True)
(DRIVE_ROOT / 'workflows').mkdir(parents=True, exist_ok=True)

if not COMFY_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/comfyanonymous/ComfyUI.git', str(COMFY_DIR)], check=True)
    subprocess.run(['pip', 'install', '-q', '-r', str(COMFY_DIR / 'requirements.txt')], check=True)

# 将 ComfyUI/models 软链到 Google Drive，模型持久化不丢失
models_link = COMFY_DIR / 'models'
if models_link.exists() and not models_link.is_symlink():
    shutil.rmtree(models_link)
if not models_link.exists():
    models_link.symlink_to(DRIVE_ROOT / 'models')

# 安装 cloudflared（免注册生成临时 HTTPS 隧道）
if not shutil.which('cloudflared'):
    subprocess.run([
        'curl', '-sL',
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb',
        '-o', '/tmp/cloudflared.deb'
    ], check=True)
    subprocess.run(['dpkg', '-i', '/tmp/cloudflared.deb'], check=True)

print('✅ 初始化完成！模型持久化目录：', DRIVE_ROOT / 'models')


In [ ]:
#@title 2. 按需下载模型到 Google Drive（已存在会自动跳过，不重复消耗流量与时间）
DOWNLOAD_ANIME_UPSCALE = True #@param {type:"boolean"}
EXTRA_MODEL_URL = "" #@param {type:"string"}
EXTRA_MODEL_SUBDIR = "checkpoints" #@param ["checkpoints", "loras", "upscale_models", "vae", "controlnet", "diffusion_models", "text_encoders"]
EXTRA_MODEL_FILENAME = "" #@param {type:"string"}

import subprocess
from pathlib import Path

MODELS_DIR = Path('/content/drive/MyDrive/AI_Video_Platform_ComfyUI/models')

def ensure_file(url: str, dest: Path):
    if dest.exists() and dest.stat().st_size > 1024:
        print(f'⏭️  已存在，跳过：{dest.name} ({dest.stat().st_size / 1024 / 1024:.1f} MB)')
        return
    dest.parent.mkdir(parents=True, exist_ok=True)
    print(f'⬇️  正在下载：{dest.name} ...')
    subprocess.run(['curl', '-fL', '-C', '-', url, '-o', str(dest)], check=True)
    print(f'✅ 下载完成：{dest.name} ({dest.stat().st_size / 1024 / 1024:.1f} MB)')

if DOWNLOAD_ANIME_UPSCALE:
    ensure_file(
        'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.2.4/RealESRGAN_x4plus_anime_6B.pth',
        MODELS_DIR / 'upscale_models' / 'RealESRGAN_x4plus_anime_6B.pth',
    )

if EXTRA_MODEL_URL.strip():
    fname = EXTRA_MODEL_FILENAME.strip() or EXTRA_MODEL_URL.strip().split('?')[0].rstrip('/').split('/')[-1]
    ensure_file(EXTRA_MODEL_URL.strip(), MODELS_DIR / EXTRA_MODEL_SUBDIR / fname)


In [ ]:
#@title 3. 启动 ComfyUI + 闲置自动断电看门狗 + Cloudflare HTTPS 入口
IDLE_SHUTDOWN_MINUTES = 10 #@param {type:"integer"}

import json
import re
import subprocess
import threading
import time
import urllib.request
from google.colab import runtime

COMFY_PORT = 8188

# 1) 启动 ComfyUI 后台进程（日志直写 /tmp/comfyui.log，避免 PIPE 阻塞并便于排错）
from pathlib import Path
if not Path('/content/ComfyUI/main.py').exists():
    raise RuntimeError('未找到 /content/ComfyUI/main.py（可能是运行时刚重连），请先重新运行 Cell 1')
log_path = Path('/tmp/comfyui.log')
log_fh = log_path.open('w', encoding='utf-8')
comfy_proc = subprocess.Popen(
    ['python3', 'main.py', '--listen', '127.0.0.1', '--port', str(COMFY_PORT)],
    cwd='/content/ComfyUI',
    stdout=log_fh,
    stderr=subprocess.STDOUT,
)

print('⏳ 等待 ComfyUI 就绪（首次加载节点可能需要 1–2 分钟）...')
for _ in range(180):
    if comfy_proc.poll() is not None:
        log_fh.flush()
        tail = '\n'.join(log_path.read_text(encoding='utf-8', errors='ignore').splitlines()[-40:])
        raise RuntimeError(f'ComfyUI 进程异常退出（code={comfy_proc.returncode}），最后日志：\n{tail}')
    try:
        with urllib.request.urlopen(f'http://127.0.0.1:{COMFY_PORT}/system_stats', timeout=2) as r:
            if r.status == 200:
                print('✅ ComfyUI 已就绪！')
                break
    except Exception:
        time.sleep(1)
else:
    log_fh.flush()
    tail = '\n'.join(log_path.read_text(encoding='utf-8', errors='ignore').splitlines()[-40:])
    raise RuntimeError(f'ComfyUI 启动超时，最后日志：\n{tail}')

# 2) 闲置自动停表看门狗：连续 IDLE_SHUTDOWN_MINUTES 分钟无新任务自动销毁运行时
def idle_watchdog():
    if IDLE_SHUTDOWN_MINUTES <= 0:
        print('ℹ️  已关闭闲置自动停表（请务必用完手动断开运行时！）')
        return
    last_active = time.monotonic()
    last_history_count = -1
    print(f'🛡️  闲置自动停表已开启：连续 {IDLE_SHUTDOWN_MINUTES} 分钟无任务将自动释放 GPU')
    while comfy_proc.poll() is None:
        time.sleep(30)
        try:
            with urllib.request.urlopen(f'http://127.0.0.1:{COMFY_PORT}/queue', timeout=3) as r:
                q = json.loads(r.read().decode())
            with urllib.request.urlopen(f'http://127.0.0.1:{COMFY_PORT}/history', timeout=3) as r:
                h = json.loads(r.read().decode())
            busy = bool(q.get('queue_running') or q.get('queue_pending'))
            h_count = len(h) if isinstance(h, dict) else 0
            if busy or (last_history_count >= 0 and h_count != last_history_count):
                last_active = time.monotonic()
            last_history_count = h_count
        except Exception:
            pass
        idle_min = (time.monotonic() - last_active) / 60.0
        if idle_min >= IDLE_SHUTDOWN_MINUTES:
            print(f'💤 已连续闲置 {idle_min:.1f} 分钟，正在自动断开运行时停表以节省计算单元...')
            comfy_proc.terminate()
            runtime.unassign()
            return

threading.Thread(target=idle_watchdog, daemon=True).start()

# 3) 启动 Cloudflare Quick Tunnel 并打印公网入口
tunnel_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', f'http://127.0.0.1:{COMFY_PORT}'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

url_re = re.compile(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com')
try:
    for line in tunnel_proc.stdout:
        m = url_re.search(line)
        if m:
            print('\n' + '=' * 64)
            print('🚀 ComfyUI 在线地址（可直接打开或填入平台引擎管理）：')
            print('   ', m.group(0))
            print('=' * 64 + '\n', flush=True)
except KeyboardInterrupt:
    print('🛑 手动停止 ComfyUI 与隧道')
finally:
    tunnel_proc.terminate()
    comfy_proc.terminate()


In [ ]:
#@title 4. 用完立刻点我：立即释放 GPU 运行时停表（0 浪费！）
from google.colab import runtime
print('🔌 正在立即断开并释放 Colab 运行时，停止计费...')
runtime.unassign()
